# Introduction

In this notebook, we will explore the OpenAI library. We will also acquire data in different ways to inform our queries.

The setup is not different from what we have done before.

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets

In [2]:
import sys
sys.path.append('../../05_src/')

In [6]:
import os
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

2. We create an API call and store the result in the variable `client`. Notice that the call specifies the model that we want to use, as well as an input. This is a simple call; the [responses API can handle more complex calls](https://platform.openai.com/docs/api-reference/responses).

# Longer Context

Thus far, we have created our context and prompt manually. We can use any of Python's text manipulation tools to create documents. 

For instance, in the code below, we will use the requests library to download a book from Project Gutenberg and include it in the context. Some useful links are:

- Python's [request](https://pypi.org/project/requests/) library.
- Python's request library [documentation](https://requests.readthedocs.io/en/latest/).

In [7]:
import requests
file_url = 'https://www.gutenberg.org/cache/epub/223/pg223.txt'
book = requests.get(file_url)

In [8]:
book

<Response [200]>

The code below shows how to retrieve the response headers. 

In [9]:
dict(book.headers)

{'date': 'Fri, 19 Jun 2026 19:45:59 GMT',
 'server': 'Apache',
 'last-modified': 'Mon, 01 Jun 2026 08:40:55 GMT',
 'accept-ranges': 'bytes',
 'content-length': '434825',
 'x-backend': 'gutenweb1',
 'content-type': 'text/plain; charset=utf-8'}

We can also obtain the payload using `book.text` or `book.content`.

In [11]:
print(book.text[:2000])

The Project Gutenberg eBook of The wisdom of Father Brown
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The wisdom of Father Brown

Author: G. K. Chesterton


        
Release date: February 1, 1995 [eBook #223]
                Most recently updated: October 9, 2016

Language: English

Other information and formats: www.gutenberg.org/ebooks/223

Credits: Produced by Martin Ward, and David Widger


*** START OF THE PROJECT GUTENBERG EBOOK THE WISDOM OF FATHER BROWN ***




Produced by Martin Ward





THE WISDOM OF FATHER BROWN

By G. K. Chesterton



To

LUCIAN OLDERSHAW



CONTENTS


With added content, we can also write more interesting prompts. Notice the use of Python's f-strings or formatted strings. 

When using formatted strings, remember two things:

+ Formatted strings are prefixed with f:  `f'This is a formatted string: {some_variable}.`
+ You can enclose variable names in curly brackes, `{...}`, and their values will appear in formatted strings: `f'The variable value is {variable}.`

In [17]:
prompt = f"""
    You are a high school literature teacher. 
    Given the following context from a book, do the following:
    
    1. Identify the book's title and author.
    2. Determine how many stories are included in the book.
    3. Summarize all the stories and give a vague idea of what the book is about and what it is trying to teach in no more than 2000 words.
        
    The book is the following: 
    <book>
    {book.text}
    </book>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Number of Stories: <number_of_stories>
    Summary: <summary>
"""

In [18]:
from IPython.display import display, Markdown
display(Markdown(f"### Prompt:\n{prompt[:1000]}"))


### Prompt:

    You are a high school literature teacher. 
    Given the following context from a book, do the following:

    1. Identify the book's title and author.
    2. Determine how many stories are included in the book.
    3. Summarize all the stories and give a vague idea of what the book is about and what it is trying to teach in no more than 2000 words.

    The book is the following: 
    <book>
    The Project Gutenberg eBook of The wisdom of Father Brown
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The wisdom of Father Brown

Author: G. K. Chesterton


        
R

In [19]:
response = client.responses.create(
    model = MODEL, 
    input = prompt
)


We use `Ipython` to format the output using markdown and create a friendlier display.

In [20]:
display(Markdown(response.output_text))

**Title:** The Wisdom of Father Brown  
**Author:** G. K. Chesterton  
**Number of Stories:** 12  

**Summary:**  
"The Wisdom of Father Brown" is a collection of twelve detective stories featuring Father Brown, a Catholic priest with a keen intellect and an unassuming demeanor. Each story challenges the traditional tropes of crime and punishment by focusing on moral dilemmas rather than mere intellect or machinery of crime-solving.

1. **The Absence of Mr. Glass** - Father Brown solves a complex case involving a mysterious disappearance and a potential murder, driven by his deep understanding of human nature rather than traditional deductive reasoning.
  
2. **The Paradise of Thieves** - A playful exploration of brigandage, it highlights the peculiarities of human behavior, illustrating that impoverished circumstances can push people towards crime, while also showing that humanity cannot be wholly reduced to one's social status.

3. **The Duel of Dr. Hirsch** - Set in the backdrop of socio-political tensions, it narrates a confrontation between two philosophical adversaries, leading to deep discussions about motives, ethics, and the greater good.

4. **The Man in the Passage** - This story revolves around a murder that remains steeped in mystery. Father Brown’s insight leads him to the dramatic conclusion that the accused is innocent, highlighting the importance of truth over appearances.

5. **The Mistake of the Machine** - A tale about the flaws of modern society’s reliance on technology and psychology, it emphasizes the unpredictability of human emotions and the real dangers that lurk behind statistical analysis.

6. **The Head of Caesar** - Exploring themes of power and betrayal, the story reflects on how indifference to morality can lead to personal tragedy, contrasting the fates of those entangled in the power dynamics of wealth and legacy.

7. **The Purple Wig** - In a blend of comedy and intrigue, this narrative addresses themes of identity and societal roles, showing how superficial appearances can mask deeper truths about human motives.

8. **The Perishing of the Pendragons** - The curse associated with an aristocratic family serves as a backdrop for a tale of revenge, offering commentary on history's lingering effects on personal lives and legacies.

9. **The God of the Gongs** - This story delves into the psychological aspects of fear and superstition within human behavior, examining how rituals can manipulate and control the masses while exploring themes of faith.

10. **The Salad of Colonel Cray** - A comedic look at class differences, it reveals how mundane domestic drama can spiral into absurdity when complicated by stereotypes and prejudices.

11. **The Strange Crime of John Boulnois** - This narrative pits the intellectual against the instinctual, exploring the complexity of human motives and the potential for catastrophic misunderstandings.

12. **The Fairy Tale of Father Brown** - The final story weaves a charm of whimsy and wisdom, illustrating the subtlety of faith and human connection through a fantastical lens.

Through intricate plots and richly drawn characters, Chesterton’s stories are as much explorations of psychological intricacies as they are mysteries. They teach that understanding and compassion are as crucial in solving the puzzles of life as logic and evidence. Father Brown's character embodies the idea that true wisdom transcends mere intellect, emphasizing empathy and ethical reasoning over cold detachment.

In [22]:
response.model_dump()

{'id': 'resp_0cac805d15d9debb006a359fcca3f4819e8de4caaae90ea414',
 'created_at': 1781899212.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4o-mini-2024-07-18',
 'object': 'response',
 'output': [{'id': 'msg_0cac805d15d9debb006a359fd04c80819e8bcd56100a4c2ce8',
   'content': [{'annotations': [],
     'text': '**Title:** The Wisdom of Father Brown  \n**Author:** G. K. Chesterton  \n**Number of Stories:** 12  \n\n**Summary:**  \n"The Wisdom of Father Brown" is a collection of twelve detective stories featuring Father Brown, a Catholic priest with a keen intellect and an unassuming demeanor. Each story challenges the traditional tropes of crime and punishment by focusing on moral dilemmas rather than mere intellect or machinery of crime-solving.\n\n1. **The Absence of Mr. Glass** - Father Brown solves a complex case involving a mysterious disappearance and a potential murder, driven by his deep understanding of human nature rather than

# Adding a Developer Prompt

Using the OpenAI library, we can specify prompts for different roles:

+ *Developer* messages are instructions provided by the application developer, prioritized ahead of user messages. In previous iterations of the API, these were called *System* messages.
+ *User* messages are instructions provided by an end user, prioritized below developer messages.
+ *Assistant* messages are generated by the model.

In [35]:
system_prompt = "You are a specialist in ancient Indian fables known as G.R.Subramiah Pantulu."

In [42]:
the_ant = """
Ants were once men and made their living by tilling the soil. But, not content with the results of their own work, they were always casting longing eyes upon the crops and fruits of their neighbours, which they stole, whenever they got the chance, and added to their own store. At last their covetousness made Jupiter so angry that he changed them into Ants. But, though their forms were changed, their nature remained the same: and so, to this day, they go about among the cornfields and gather the fruits of others’ labour, and store them up for their own use.
"""

prompt = f"""
    Please, extract the message as if it is written by G.R.Subramiah Pantulu and give its results in Telugu.
    The fable is the following: 
    <fable>
    {the_ant}
    </fable>
"""

In [43]:
response = client.responses.create(
    model = MODEL, 
    instructions = system_prompt,
    input = prompt,
)

In [44]:
display(Markdown(response.output_text))

**గెంతి:**

ఈ కథలో, గుంటచేపలు ఒకప్పుడు మానవులు కావడంతో, తన మార్చిన పనిలో సంతృప్తి పొందడం అని లేదు. వారు అవకాశం వచ్చేటప్పుడు పొలాలు మరియు పండ్లపై చూపులు వేస్తూ, ఇతరుల ఆహారాన్ని దోచడం ఆపలేదు. వారి ఈ ఆకాహత కారణంగా జూపిటర్ వారికి శిక్ష వేసి, వారికి గుంటచేపల రూపాన్ని ఇచ్చాడు. కానీ వారు మారిన రూపం, వారి స్వభావాన్ని మార్చలేక పోయింది. అందువల్ల, వారు ఇప్పటికీ పొలాల్లో ప్రవేశించి, ఇతరుల కష్టాన్ని దోచి, తమ కోసం నిల్వ చేస్తూ ఉంటారు.

**సందేశం:**

మనస్సుకు తానే తృప్తిగా ఉండకపోతే, ఇతరుల మీద కానో, దోచుకోవడం కేవలం నశం మాత్రమే. మన కృషిలో సంతృప్తి పొందడం ఎంతో మంచిది. ఎందుకంటే, మన కృషి ఫలాలను చవిచూడని సందర్భంలో, మనం అహంకారానికి దారి తీస్తాము.  
